In [1]:
from google.colab import files
uploaded = files.upload()
for filename in uploaded.keys():
    print("Uploaded file:", filename)


In [ ]:
# !rm -f index.html

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# for filename in uploaded.keys():
#     print("Uploaded file:", filename)


In [ ]:
# !pkill ngrok

In [ ]:
# !pip install flask flask-ngrok flask-cors pyngrok tensorflow
!pip install flask flask-ngrok flask-cors pyngrok
!pip install tensorflow==2.19.0

In [ ]:
!python --version

Python 3.12.11


In [ ]:
import nest_asyncio
nest_asyncio.apply()

from flask import Flask, request, render_template
from flask_cors import CORS
from pyngrok import ngrok
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from breed_data import breed_info  # breed info dictionary
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

In [ ]:
# Load base model
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
base_model.trainable = False

# Rebuild Sequential model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(5, activation="softmax")  # replace 5 with your number of classes
])


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
model.load_weights("cattle_breed_model.h5")
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,597 (9.24 MB)

 Trainable params: 164,613 (643.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
# 🔹 Load model and labels
# model = load_model("cattle_breed_model.h5", custom_objects={"MobileNet": MobileNetV2})
# model = load_model("cattle_breed_model.h5", compile=False)

with open("labels.txt") as f:
    labels = [line.strip() for line in f.readlines()]

app = Flask(__name__, template_folder=".")  # templates in current folder
CORS(app)

# 🔹 Routes for frontend pages
@app.route("/")
def index():
    return render_template("index.html")

@app.route("/myfarm")
def myfarm():
    return render_template("myfarm.html")

# 🔹 Prediction API
@app.route("/predict", methods=["POST"])
def predict():
    if "file" not in request.files:
        return {"error":"No file uploaded"}

    file = request.files["file"]
    file.save("uploaded.jpg")

    img = image.load_img("uploaded.jpg", target_size=(224,224))
    x = image.img_to_array(img)
    x = np.expand_dims(x,0)/255.0

    preds = model.predict(x)
    idx = np.argmax(preds[0])
    breed = labels[idx]
    confidence = round(float(np.max(preds[0]))*100,2)

    # Get milk & health info
    info = breed_info.get(breed, {"milk_production": "N/A", "health_tips": "N/A"})

    return {
        "breed": breed,
        "confidence": confidence,
        "milk_production": info["milk_production"],
        "health_tips": info["health_tips"]
    }

# 🔹 Start ngrok tunnel
ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")
public_url = ngrok.connect(8000)
print("🚀 Open this URL in browser:", public_url)

# 🔹 Run Flask
app.run(host="0.0.0.0", port=8000)

🚀 Open this URL in browser: NgrokTunnel: "https://368af53be5e2.ngrok-free.app" -> "http://localhost:8000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://172.28.0.12:8000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [13/Sep/2025 15:17:48] "GET / HTTP/1.1" 200 -
